In [1]:
import pandas as pd

In [2]:
# 2 ETL and Cleaning
# 2.1 Make Relational Data - Merge 3 tables 
df_telemetry = pd.read_csv('aviation_data/fact_telemetry_high_freq.csv')
df_summary = pd.read_csv('aviation_data/flight_summary.csv')
df_registry = pd.read_csv('aviation_data/dim_aircraft.csv')

master_df = df_telemetry.merge(df_summary, on='flight_id', how='left') \
                        .merge(df_registry, on='tail_number', how='left')

master_df.head()

,flight_id,timestamp_sec,altitude_ft,battery_soc_pct,phase,tail_number,payload_lbs,ambient_temp_c,freight_rate_per_lb,ticket_revenue,model,battery_capacity_kwh,max_payload_lbs,firmware_version
0,FL-00001,0,0.0,99.97,CLIMB,N108AL,1249.080238,-3.428541,2.5,3122.700594,Alice_V1,850,2500,v2.1
1,FL-00001,1,33.3,99.94,CLIMB,N108AL,1249.080238,-3.428541,2.5,3122.700594,Alice_V1,850,2500,v2.1
2,FL-00001,2,66.6,99.91,CLIMB,N108AL,1249.080238,-3.428541,2.5,3122.700594,Alice_V1,850,2500,v2.1
3,FL-00001,3,99.9,99.88,CLIMB,N108AL,1249.080238,-3.428541,2.5,3122.700594,Alice_V1,850,2500,v2.1
4,FL-00001,4,133.2,99.85,CLIMB,N108AL,1249.080238,-3.428541,2.5,3122.700594,Alice_V1,850,2500,v2.1


In [3]:
# 2.2 Cleaning & Tagging    
# 1. Temporal Continuity: Fill missing sensor readings with the last known valid state
# This assumes the system state doesn't change drastically within 1-second intervals.
master_df['battery_soc_pct'] = master_df['battery_soc_pct'].ffill()

# 2. Force battery SoC to stay within physical limits [0, 100]
master_df['battery_soc_pct'] = master_df['battery_soc_pct'].clip(0, 100)

# 3. Altitude cannot be negative (we assume ground level is 0)
master_df['altitude_ft'] = master_df['altitude_ft'].clip(lower=0)

# 4. Tag safety compliance (20% FAA Reserve Mandate)
master_df['is_alert'] = master_df['battery_soc_pct'] < 20.0 

print(f"Total Telemetry records: {len(master_df):,}")
print(f"Flights with at least one safety alert: {master_df[master_df['is_alert'] == True]['flight_id'].nunique()}")
master_df.tail(5)


Total Telemetry records: 442,297
Flights with at least one safety alert: 100


,flight_id,timestamp_sec,altitude_ft,battery_soc_pct,phase,tail_number,payload_lbs,ambient_temp_c,freight_rate_per_lb,ticket_revenue,model,battery_capacity_kwh,max_payload_lbs,firmware_version,is_alert
442292,FL-00100,4495,0.0,12.40,LANDING,N103AL,715.782854,33.993777,2.5,1789.457135,Alice_V1,850,2500,v2.2,True
442293,FL-00100,4496,0.0,12.39,LANDING,N103AL,715.782854,33.993777,2.5,1789.457135,Alice_V1,850,2500,v2.2,True
442294,FL-00100,4497,0.0,12.38,LANDING,N103AL,715.782854,33.993777,2.5,1789.457135,Alice_V1,850,2500,v2.2,True
442295,FL-00100,4498,0.0,12.37,LANDING,N103AL,715.782854,33.993777,2.5,1789.457135,Alice_V1,850,2500,v2.2,True
442296,FL-00100,4499,0.0,12.36,LANDING,N103AL,715.782854,33.993777,2.5,1789.457135,Alice_V1,850,2500,v2.2,True


In [4]:
# 2.3 Downsampling to 1-minute granularity
# Logic: Aggregating 1Hz telemetry to 1-min to focus on operational trends
master_df['timestamp_min'] = (master_df['timestamp_sec'] // 60) 

# Define aggregation strategy
agg_strategy = {
    'battery_soc_pct': 'mean',
    'altitude_ft': 'mean',
    'payload_lbs': 'first',        # Flight-level constant
    'ambient_temp_c': 'first',     # Flight-level constant
    'ticket_revenue': 'first',     # Flight-level constant
    'firmware_version': 'first',   # Flight-level constant
    'is_alert': 'max',             # If alert occurred at any second in a minute, mark as True
    'phase': 'first' 
}

analytical_view = master_df.groupby(['flight_id', 'timestamp_min']).agg(agg_strategy).reset_index()

# Audit
print(f"Downsampling: {len(analytical_view):,} rows")
print(f"Compression Ratio: {((1 - len(analytical_view)/len(master_df)) * 100):.1f}%")
analytical_view.head(10)

Downsampling: 7,387 rows
Compression Ratio: 98.3%


,flight_id,timestamp_min,battery_soc_pct,altitude_ft,payload_lbs,ambient_temp_c,ticket_revenue,firmware_version,is_alert,phase
0,FL-00001,0,99.074500,982.350,1249.080238,-3.428541,3122.700594,v2.1,False,CLIMB
1,FL-00001,1,97.254167,2980.350,1249.080238,-3.428541,3122.700594,v2.1,False,CLIMB
2,FL-00001,2,95.433667,4978.350,1249.080238,-3.428541,3122.700594,v2.1,False,CLIMB
3,FL-00001,3,93.613167,6976.350,1249.080238,-3.428541,3122.700594,v2.1,False,CLIMB
4,FL-00001,4,91.792833,8974.350,1249.080238,-3.428541,3122.700594,v2.1,False,CLIMB
5,FL-00001,5,89.972333,10972.350,1249.080238,-3.428541,3122.700594,v2.1,False,CLIMB
6,FL-00001,6,88.151833,12970.350,1249.080238,-3.428541,3122.700594,v2.1,False,CLIMB
7,FL-00001,7,86.385833,14734.425,1249.080238,-3.428541,3122.700594,v2.1,False,CLIMB
8,FL-00001,8,84.935000,15000.000,1249.080238,-3.428541,3122.700594,v2.1,False,CRUISE
9,FL-00001,9,83.535000,15000.000,1249.080238,-3.428541,3122.700594,v2.1,False,CRUISE


In [5]:
# Table 4: Alice_Master_Analysis_View
analytical_view.to_csv('aviation_data/Alice_Master_Analysis_View.csv', index=False)